# RNA Score Plots

Generates two figures for datasets with RNA scores:
- **Scatter plot:** RNA score vs. SGE fitness score, colored by consequence, with optional RNA threshold line.
- **Stem plot** *(optional, requires RNA threshold)*: RNA score per CDS position faceted by exon, highlighting RNA-low variants.

**Required data:** `scores` and `thresholds` sheets from the pipeline output Excel file (`*_data.xlsx`). The `scores` sheet must contain an `RNA_score` column. The stem plot additionally requires `functional_consequence` (populated when the pipeline was run with `--rna-threshold`).

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG/SVG require `vl-convert-python` (`pip install vl-convert-python`). Change the extension to `.html` for interactive output (no extra package needed).

In [ ]:
import pandas as pd
from sgeviz.figures import rna_score

In [ ]:
# --- Configuration ---
excel_path = "GENE_data.xlsx"         # path to pipeline output Excel file
gene = "GENE"                         # gene name for plot title
# rna_threshold: set to a float to draw the RNA threshold line and enable the stem plot.
# If the pipeline was run with --rna-threshold, this value is stored in the thresholds sheet.
# Set to None to skip the stem plot and omit the threshold line from the scatter.
rna_threshold = None
scatter_output = f"{gene}_rna_score_scatter.png"  # change to .html for interactive output, or .svg
stem_output = f"{gene}_rna_stem_plot.png"

# --- Plot customization (optional) ---
width = 400                    # scatter plot width in pixels
height = 300                   # scatter plot height in pixels
score_domain = (-0.6, 0.3)    # fitness score y-axis range (min, max)
rna_domain = (-8, 3)           # RNA score x-axis range (min, max)
stem_width = 600               # stem plot panel width in pixels
stem_height = 150              # stem plot panel height in pixels

In [ ]:
# --- Load data ---
scores_df = pd.read_excel(excel_path, sheet_name="scores")
thresh_df = pd.read_excel(excel_path, sheet_name="thresholds")

thresholds = [
    thresh_df["non_functional_threshold"].iloc[0],
    thresh_df["functional_threshold"].iloc[0],
]

# Read rna_threshold from Excel if not set manually
if rna_threshold is None and "rna_threshold" in thresh_df.columns:
    val = thresh_df["rna_threshold"].iloc[0]
    if pd.notna(val):
        rna_threshold = float(val)
        print(f"RNA threshold loaded from Excel: {rna_threshold}")

if "RNA_score" not in scores_df.columns:
    raise ValueError("No 'RNA_score' column found in scores sheet.")

print(f"Loaded {len(scores_df)} variants. Thresholds: {thresholds}, RNA threshold: {rna_threshold}")

In [ ]:
# --- Scatter plot ---
scatter_chart = rna_score.make_scatter(
    scores_df, thresholds, rna_threshold=rna_threshold, gene=gene,
    width=width, height=height, score_domain=score_domain, rna_domain=rna_domain,
)
scatter_chart

In [ ]:
# --- Stem plot (requires rna_threshold and functional_consequence column) ---
if rna_threshold is None:
    print("Set rna_threshold above to generate the stem plot.")
elif "functional_consequence" not in scores_df.columns:
    print(
        "No 'functional_consequence' column found. "
        "Re-run the pipeline with --rna-threshold to populate this column."
    )
else:
    stem_chart = rna_score.make_stem_plot(
        scores_df, rna_threshold, gene=gene,
        width=stem_width, height=stem_height, rna_domain=rna_domain,
    )
    stem_chart

In [ ]:
# --- Save ---
scatter_chart.save(scatter_output)
print(f"Saved: {scatter_output}")

if rna_threshold is not None and "functional_consequence" in scores_df.columns:
    stem_chart.save(stem_output)
    print(f"Saved: {stem_output}")